# FlyBot Sentinel: RF-DETR Nano fruit-fly detector (Apache-2.0) on a Colab T4

Optional path (the primary detector is YOLO26n, AGPL-3.0; YOLOv7-tiny, GPL-3.0, is the baseline). Trains **RF-DETR Nano**
(`rfdetr` 1.11.2, Apache-2.0 code and weights) on the CamOS dataset
[camos/fruit-fly-wmcqi v3](https://universe.roboflow.com/camos/fruit-fly-wmcqi/dataset/3)
(**CC BY 4.0**), from the same **YOLOv8 zip** as the other notebooks (`rfdetr` reads YOLO
`data.yaml` folders directly, no COCO export needed), tests on the 76-image test split and
exports ONNX for OpenCV 5 DNN.

1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Run all, and pick `fruit-fly.v3i.yolov8.zip` when asked.
3. The last cell downloads `rfdetr-nano-fruitfly.onnx`, `rfdetr-nano-metrics.csv` and
   `rfdetr-nano-run.zip`. Put them in `models/rfdetr/` (see `models/COLAB_RUNBOOK.md`).

Training time on a T4 was not measured: read the first epoch's time and multiply by EPOCHS.

In [ ]:
# Settings
SIZE = 'nano'                 # 'nano' (384 px) or 'small' (512 px, the production family)
EPOCHS = 50
BATCH = 8
GRAD_ACCUM = 2                # effective batch 16
DATASET_ZIP = '/content/fruit-fly.v3i.yolov8.zip'
SMOKE_TEST = False            # True: 1 epoch on 32 images, to check the notebook end to end
RFDETR = '1.11.2'

In [ ]:
!nvidia-smi -L
!pip -q install "rfdetr[train,onnx]=={RFDETR}"

In [ ]:
# Dataset: upload if missing, unzip, polygon rows -> boxes (31 train files, 3 mixed), absolute data.yaml
import os
import shutil
from pathlib import Path

import yaml

if not os.path.exists(DATASET_ZIP):
    from google.colab import files
    up = files.upload()
    DATASET_ZIP = '/content/' + next(iter(up))
DATA = Path('/content/data')
if not (DATA / 'train').exists():
    shutil.unpack_archive(DATASET_ZIP, DATA)


def polygon_rows_to_boxes(label_dir):
    rows = 0
    for txt in Path(label_dir).glob('*.txt'):
        out, changed = [], False
        for line in txt.read_text().splitlines():
            p = line.split()
            if len(p) >= 7 and len(p) % 2 == 1:
                xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
                x0, x1, y0, y1 = max(0, min(xs)), min(1, max(xs)), max(0, min(ys)), min(1, max(ys))
                line = f'{p[0]} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}'
                rows, changed = rows + 1, True
            out.append(line)
        if changed:
            txt.write_text('\n'.join(out) + '\n')
    return rows


for split in ['train', 'valid', 'test']:
    print(split, len(list((DATA / split / 'images').glob('*'))), 'images,',
          polygon_rows_to_boxes(DATA / split / 'labels'), 'polygon rows -> boxes')
root = DATA
if SMOKE_TEST:
    root = Path('/content/data-small')
    for split in ['train', 'valid', 'test']:
        for kind in ['images', 'labels']:
            (root / split / kind).mkdir(parents=True, exist_ok=True)
        for img in sorted((DATA / split / 'images').iterdir())[:32]:
            shutil.copy(img, root / split / 'images')
            lab = DATA / split / 'labels' / (img.stem + '.txt')
            if lab.exists():
                shutil.copy(lab, root / split / 'labels')
(root / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(root), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
    'nc': 1, 'names': ['fruit_fly']}, sort_keys=False))
print((root / 'data.yaml').read_text())

In [ ]:
# Train; the best checkpoint is scored on the test split at the end (run_test)
from rfdetr import RFDETRNano, RFDETRSmall

OUT = Path(f'/content/rfdetr-{SIZE}')
model = {'nano': RFDETRNano, 'small': RFDETRSmall}[SIZE]()
model.train(dataset_dir=str(root), epochs=1 if SMOKE_TEST else EPOCHS, batch_size=BATCH,
            grad_accum_steps=GRAD_ACCUM, output_dir=str(OUT), run_test=True)
print(sorted(p.name for p in OUT.iterdir()))

In [ ]:
# ONNX for OpenCV 5 DNN (opset 17, static, batch 1) from the best checkpoint
best = next(p for p in [OUT / 'checkpoint_best_total.pth', OUT / 'checkpoint_best_ema.pth',
                        OUT / 'checkpoint_best_regular.pth'] if p.exists())
print('best checkpoint:', best)
trained = {'nano': RFDETRNano, 'small': RFDETRSmall}[SIZE](pretrain_weights=str(best))
onnx_path = trained.export(output_dir=str(OUT / 'onnx'), opset_version=17)
shutil.copy(onnx_path, f'/content/rfdetr-{SIZE}-fruitfly.onnx')
print(onnx_path)

In [ ]:
# Download: ONNX, metrics.csv (per-epoch validation and the final test mAP), and the run folder.
# The OpenCV 5 ENGINE_NEW check runs locally: uv run python models/rfdetr_probe.py load <onnx> <image>
from google.colab import files

shutil.copy(OUT / 'metrics.csv', f'/content/rfdetr-{SIZE}-metrics.csv')
shutil.make_archive(f'/content/rfdetr-{SIZE}-run', 'zip', OUT)
for f in [f'rfdetr-{SIZE}-fruitfly.onnx', f'rfdetr-{SIZE}-metrics.csv', f'rfdetr-{SIZE}-run.zip']:
    files.download('/content/' + f)